In [1]:
import pandas as pd
import numpy as np


# Mappings

In [2]:
manual_mappings = {
    "Climate Change": [
        "bio-fuel", "biofuel_production", "carbon", "carbon_pricing", "carbon_tax", 
        "climate_change", "climate_change_mitigation", "climate-related", "climatic", 
        "conserve_energy", "cop", "drought", "emission", "energy_conservation", 
        "energy_intensity", "energy-efficient", "energy-saving", "extreme_weather", 
        "global_warming", "low-carbon", "tackling_climate_change", 
        "green_energy", "net_zero", "climate_resilience", "renewable_energy"
    ],
    
    "Economic Growth": [
        "growth", "grown_faster", "growth-boosting", "growth-enhancing", 
        "growth-oriented", "growth-promoting", "growth-supporting", 
        "growth-critical", "growth-inducing", "growth-friendly", 
        "growth-friendly_manner", "pro-growth", "private_sector-led_growth",
        "economic_expansion", "sustainable_growth", "productivity_growth", 
        "boost_growth", "support_growth", "inclusive_growth", 
        "broad-based_growth", "development", "investment-led_growth", 
        "accelerated_growth", "macroeconomic_growth", "growth_potential"
    ],
    
    "Debt": [
        "debt", "bond_issuance", "bonded_debt", "debt_overhang", "debt_servicing", 
        "debt-to-gdp_ratio", "debt_sustainability", "debt-management", "debt_related", 
        "debt_service", "heavily_indebted", "high_debt", "public_debt", 
        "external_debt", "sovereign_debt", "nonconcessional_borrowing", 
        "sovereign_bond", "sovereign_default", "syndicated_loan", 
        "debt_burden", "paris_club", "fiscal_debt", "debt_ratio", 
        "indebtedness", "creditor"
    ],
    
    "Crisis": [
        "crisis", "banking_crisis", "financial_crisis", "economic_crisis", 
        "humanitarian_crisis", "sovereign_debt_crisis", "systemic_crisis", 
        "currency_crisis", "balance_of_payments_crisis", "debt_crisis", 
        "crisis_management", "crisis_response", "crisis_resolution", 
        "crisis_situations", "crisis_impact", "emergency_measures", 
        "shock_response", "policy_crisis", "crisis_aid", "fiscal_crisis", 
        "global_crisis", "contagion", "crisis_prevention", "market_disruption", 
        "crisis_fund"
    ],
    
    "Risk": [
        "risk", "financial_risks", "risk_management", "risk_mitigation", 
        "systemic_risk", "credit_risk", "market_risk", "economic_risk", 
        "operational_risk", "vulnerability", "macroeconomic_risk", 
        "risk_assessment", "geopolitical_risk", "policy_uncertainty", 
        "exposure", "risk_factors", "fiscal_risks", "tail_risks", 
        "debt_risks", "sovereign_risk", "volatility", "risk_response", 
        "risk_fund", "risk_governance", "uncertainty"
    ],
    
    "Reform": [
        "reform", "structural_reforms", "unfinished_reform_agenda", 
        "structural_reform_agenda", "deregulation", "deregulating", 
        "bankruptcy_code", "bankruptcy_law", "reduce_red_tape", 
        "cut_red_tape", "governance_reforms", "institutional_reform", 
        "policy_reform", "legal_reform", "economic_reform", "tax_reform", 
        "fiscal_reform", "market_reform", "regulatory_reform", 
        "labor_market_reform", "public_sector_reform", "civil_service_reform", 
        "judicial_reform", "transparency_reforms", "anti_corruption_reform"
    ]
}


# Constituency Statements

In [3]:
data=pd.read_csv(r"csvs\Topic_Distribution_Unequal.csv")

In [4]:
df=data[data.columns[0:8]]

In [5]:
import ast

# Safely convert string representations to Python objects
df['Bigrams'] = df['Bigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['Trigrams'] = df['Trigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['Processed_Text'] = df['Processed_Text'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


C:\Users\Husnain\AppData\Local\Temp\ipykernel_20380\4044632308.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Bigrams'] = df['Bigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
C:\Users\Husnain\AppData\Local\Temp\ipykernel_20380\4044632308.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Trigrams'] = df['Trigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
C:\Users\Husnain\AppData\Local\Temp\ipykernel_20380\4044632308.py:6: SettingWithCopyWarn

In [6]:
# Step 3: Count Topic Appearances in Each Document
def count_topic_appearances(text_tokens, bigrams, trigrams, lexicon):
    topic_counts = {topic: 0 for topic in lexicon.keys()}

    for topic, terms in lexicon.items():
        for term in terms:
            term_parts = term.split("_")  # Split term into words
            
            if len(term_parts) == 1:  # Single-word terms
                topic_counts[topic] += text_tokens.count(term)

            elif len(term_parts) == 2:  # Bigram terms
                topic_counts[topic] += bigrams.count(tuple(term_parts))

            elif len(term_parts) == 3:  # Trigram terms
                topic_counts[topic] += trigrams.count(tuple(term_parts))
    
    return topic_counts

# Apply function
df['Topic_Counts'] = df.apply(lambda row: count_topic_appearances(row['Processed_Text'], row['Bigrams'], row['Trigrams'], manual_mappings), axis=1)

# Convert dictionary column into separate columns
topic_counts_df = pd.DataFrame(df['Topic_Counts'].tolist(), index=df.index)

# Step 4: Compute Total Topic Count Per Document
df['Total_Topic_Count'] = topic_counts_df.sum(axis=1)

# Step 5: Compute Topic Distribution (Normalized Frequency)
topic_distribution = topic_counts_df.div(df['Total_Topic_Count'], axis=0).fillna(0)

# Step 6: Merge Topic Distribution with Original Data
df = pd.concat([df, topic_distribution], axis=1)
df[['Title', 'Year'] + list(topic_distribution.columns)]


C:\Users\Husnain\AppData\Local\Temp\ipykernel_20380\71922568.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Topic_Counts'] = df.apply(lambda row: count_topic_appearances(row['Processed_Text'], row['Bigrams'], row['Trigrams'], manual_mappings), axis=1)
C:\Users\Husnain\AppData\Local\Temp\ipykernel_20380\71922568.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Total_Topic_Count'] = topic_counts_df.sum(axis=1)


,Title,Year,Climate Change,Economic Growth,Debt,Crisis,Risk,Reform
0,"IMFC Statement by Christine Lagarde, President...",2024,0.064516,0.387097,0.032258,0.032258,0.483871,0.000000
1,"IMFC Statement by HE Haitham Al Ghais, Secreta...",2024,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
2,"IMFC Statement by Ayman Al-Sayari, Governor of...",2024,0.000000,0.346154,0.365385,0.153846,0.096154,0.038462
3,"IMFC Statement by Antoine Armand, Minister of ...",2024,0.125000,0.312500,0.250000,0.000000,0.187500,0.125000
4,"IMFC Statement by Luis Caputo, Minister of Eco...",2024,0.051546,0.432990,0.360825,0.030928,0.061856,0.061856
...,...,...,...,...,...,...,...,...
559,IMFC Statement by the Honorable Domenico Sinis...,2004,0.000000,0.447368,0.328947,0.105263,0.013158,0.105263
560,"IMFC Statement by the Honorable John W. Snow, ...",2004,0.000000,0.466667,0.366667,0.000000,0.033333,0.133333
561,IMFC Statement by H.E. Sadakazu Tanigaki Minis...,2004,0.000000,0.275862,0.275862,0.206897,0.034483,0.206897
562,"IMFC Statement By James D. Wolfensohn, Preside...",2004,0.000000,0.630435,0.217391,0.000000,0.065217,0.086957


In [8]:
df.to_csv(r"csvs\Topic_Distribution_equal.csv", index=False)

# Comuniques

In [14]:
data_c=pd.read_csv(r"csvs\Topic_Distribution_Unequal_comuniques.csv")

In [15]:
data_c=data_c[data_c.columns[0:8]]

In [6]:
data_c

,Title,Date,Subtext,Processed_Text,Bigrams,Trigrams,Topic_Counts,Total_Topic_Count
0,Intergovernmental Group of Twenty-Four on Inte...,"October 22, 2024",1. The G-24 expresses its deep concern over th...,"['expresses', 'deep', 'concern', 'humanitarian...","[('expresses', 'deep'), ('deep', 'concern'), (...","[('expresses', 'deep', 'concern'), ('deep', 'c...","{'Climate Change': 4, 'Economic Growth': 8, 'D...",196
1,Intergovernmental Group of Twenty-Four on Inte...,"April 16, 2024",1. The G‑24 recognizes the profound human suff...,"['recognizes', 'profound', 'human', 'suffering...","[('recognizes', 'profound'), ('profound', 'hum...","[('recognizes', 'profound', 'human'), ('profou...","{'Climate Change': 2, 'Economic Growth': 5, 'D...",209
2,Intergovernmental Group of Twenty-Four on Inte...,"October 10, 2023",1. We express our condolenceson...,"['express', 'condolenceson', 'human', 'sufferi...","[('express', 'condolenceson'), ('condolenceson...","[('express', 'condolenceson', 'human'), ('cond...","{'Climate Change': 2, 'Economic Growth': 5, 'D...",150
3,Intergovernmental Group of Twenty-Four on Inte...,"April 11, 2023",Ministers of the Intergovernmental Group of Tw...,"['ministers', 'intergovernmental', 'group', 'i...","[('ministers', 'intergovernmental'), ('intergo...","[('ministers', 'intergovernmental', 'group'), ...","{'Climate Change': 0, 'Economic Growth': 0, 'D...",8
4,Intergovernmental Group of Twenty-Four on Inte...,"October 11, 2022",1. Multiple compounding crises have severely d...,"['multiple', 'compounding', 'crises', 'severel...","[('multiple', 'compounding'), ('compounding', ...","[('multiple', 'compounding', 'crises'), ('comp...","{'Climate Change': 8, 'Economic Growth': 7, 'D...",199
...,...,...,...,...,...,...,...,...
168,Communiqué of the Interim Committee of the Boa...,"September 21, 1997","In the advanced economies as a group, growth ...","['advanced', 'economies', 'group', 'growth', '...","[('advanced', 'economies'), ('economies', 'gro...","[('advanced', 'economies', 'group'), ('economi...","{'Climate Change': 0, 'Economic Growth': 6, 'D...",99
169,Group of Twenty Four Communiqué,"September 20, 1997",Ministers of the Intergovernmental Group of Tw...,"['ministers', 'intergovernmental', 'group', 'i...","[('ministers', 'intergovernmental'), ('intergo...","[('ministers', 'intergovernmental', 'group'), ...","{'Climate Change': 0, 'Economic Growth': 34, '...",1379
170,Communiqué of the Ministers and Governors of t...,"April 28, 1997","Washington, D.C. 1. The Ministers and Central ...","['washington', 'ministers', 'central', 'bank',...","[('washington', 'ministers'), ('ministers', 'c...","[('washington', 'ministers', 'central'), ('min...","{'Climate Change': 0, 'Economic Growth': 0, 'D...",170
171,Interim Committee Communiqué,"April 28, 1997",The Committee welcomed the generally favorable...,"['committee', 'welcomed', 'generally', 'favora...","[('committee', 'welcomed'), ('welcomed', 'gene...","[('committee', 'welcomed', 'generally'), ('wel...","{'Climate Change': 0, 'Economic Growth': 29, '...",837


In [11]:
import ast

# Safely convert string representations to Python objects
df['Bigrams'] = df['Bigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['Trigrams'] = df['Trigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['Processed_Text'] = df['Processed_Text'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


In [16]:
data=data_c.copy()

In [18]:
# Step 3: Count Topic Appearances in Each Document
def count_topic_appearances(text_tokens, bigrams, trigrams, lexicon):
    topic_counts = {topic: 0 for topic in lexicon.keys()}

    for topic, terms in lexicon.items():
        for term in terms:
            term_parts = term.split("_")  # Split term into words
            
            if len(term_parts) == 1:  # Single-word terms
                topic_counts[topic] += text_tokens.count(term)

            elif len(term_parts) == 2:  # Bigram terms
                topic_counts[topic] += bigrams.count(tuple(term_parts))

            elif len(term_parts) == 3:  # Trigram terms
                topic_counts[topic] += trigrams.count(tuple(term_parts))
    
    return topic_counts

# Apply function
# First, ensure text columns are properly converted from string to Python objects
data['Bigrams'] = data['Bigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
data['Trigrams'] = data['Trigrams'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
data['Processed_Text'] = data['Processed_Text'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Then apply the topic counting function
data['Topic_Counts'] = data.apply(lambda row: count_topic_appearances(row['Processed_Text'], row['Bigrams'], row['Trigrams'], manual_mappings), axis=1)

# Convert dictionary column into separate columns
topic_counts_df = pd.DataFrame(data['Topic_Counts'].tolist(), index=data.index)

# Step 4: Compute Total Topic Count Per Document
data['Total_Topic_Count'] = topic_counts_df.sum(axis=1)

# Step 5: Compute Topic Distribution (Normalized Frequency)
topic_distribution = topic_counts_df.div(data['Total_Topic_Count'], axis=0).fillna(0)

# Step 6: Merge Topic Distribution with Original Data
data = pd.concat([data, topic_distribution], axis=1)
data[['Title', 'Date'] + list(topic_distribution.columns)]


,Title,Date,Climate Change,Economic Growth,Debt,Crisis,Risk,Reform
0,Intergovernmental Group of Twenty-Four on Inte...,"October 22, 2024",0.063492,0.444444,0.301587,0.031746,0.079365,0.079365
1,Intergovernmental Group of Twenty-Four on Inte...,"April 16, 2024",0.035714,0.392857,0.500000,0.035714,0.000000,0.035714
2,Intergovernmental Group of Twenty-Four on Inte...,"October 10, 2023",0.045455,0.454545,0.340909,0.000000,0.022727,0.136364
3,Intergovernmental Group of Twenty-Four on Inte...,"April 11, 2023",0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
4,Intergovernmental Group of Twenty-Four on Inte...,"October 11, 2022",0.126984,0.412698,0.317460,0.063492,0.031746,0.047619
...,...,...,...,...,...,...,...,...
168,Communiqué of the Interim Committee of the Boa...,"September 21, 1997",0.000000,0.800000,0.000000,0.000000,0.000000,0.200000
169,Group of Twenty Four Communiqué,"September 20, 1997",0.000000,0.281984,0.511749,0.161880,0.015666,0.028721
170,Communiqué of the Ministers and Governors of t...,"April 28, 1997",0.000000,0.758621,0.000000,0.000000,0.241379,0.000000
171,Interim Committee Communiqué,"April 28, 1997",0.000000,0.830846,0.000000,0.000000,0.004975,0.164179


In [20]:
data.to_csv(r"csvs\Topic_Distribution_equal_comuniques.csv", index=False)